# Week 10 – Strategy Comparison

For every ticker in `TICKERS`, compares:
1. Buy-and-hold
2. SMA crossover (50/200)
3. LLM-guided strategy (Week 8 simulation)

Writes a markdown report to `reports/evaluation_report.md`.

**Caveat:** the LLM strategy is scored on *realised* P&L of closed trades only (open positions are not marked to market), so compare it to the baselines with care. The LLM simulation runs once per ticker and can be slow.

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Project root = nearest parent folder that contains src/ (works from any cwd).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

# ---- Tickers to analyse (edit this list) -----------------------------------
TICKERS = ["AAPL", "MSFT", "GOOGL"]

from zipline import run_algorithm
from src.evaluation.csvdir_prep import prepare_csvdir
from src.evaluation.zipline_baselines import (
    ensure_bundle, make_buy_and_hold, make_sma_crossover, trade_pnls,
    utc, show_error,
)
from src.evaluation.metrics import compute_sharpe, compute_max_drawdown, compute_win_rate
from src.simulation.orchestrator import simulate_historical
# ---- Data -------------------------------------------------------------------
# A 50/200-day SMA needs years of history, but data/processed/indicators only
# holds ~60 trading days per ticker. So long Yahoo history is downloaded once
# into <root>/data/csvdir/daily/ (cached afterwards).
# Set SOURCE = "indicators" to convert your indicator CSVs instead (short windows only).
DATA_START, DATA_END = "2014-01-01", "2024-12-31"
SOURCE = "yahoo"                       # or "indicators"
CSV_DIR = str(ROOT / "data" / "csvdir")
INDICATORS_DIR = str(ROOT / "data" / "processed" / "indicators")

prepare_csvdir(TICKERS, CSV_DIR, source=SOURCE, start=DATA_START, end=DATA_END,
               indicators_dir=INDICATORS_DIR)
BUNDLE = ensure_bundle(CSV_DIR)        # re-ingests automatically if the tickers change

In [ ]:
start_date = "2026-03-01"
end_date   = "2026-10-05"   # last 2026 trading day
starting_capital = 10_000.0

print(f"Running comparison for {TICKERS} from {start_date} to {end_date}...")

In [ ]:
def run(strategy):
    initialize, handle_data = strategy
    return run_algorithm(
        start=utc(start_date), end=utc(end_date),
        initialize=initialize, handle_data=handle_data,
        capital_base=starting_capital, bundle=BUNDLE,
    )

baselines = {}     # ticker -> {"Buy-and-hold": result, "SMA 50/200": result}
for t in TICKERS:
    try:
        baselines[t] = {
            "Buy-and-hold": run(make_buy_and_hold(t)),
            "SMA 50/200":   run(make_sma_crossover(t, fast=50, slow=200)),
        }
    except Exception as exc:
        show_error(f"{t} baselines", exc)

In [ ]:
llm_sims = {}
for t in baselines:                      # only tickers whose baselines ran
    print(f"Running LLM-guided simulation for {t}...")
    try:
        llm_sims[t] = simulate_historical(
            ticker=t, start_date=start_date, end_date=end_date,
            portfolio_value=starting_capital, verbose=False,
        )
    except Exception as exc:
        print(f"{t}: LLM simulation skipped ({type(exc).__name__}: {exc})")

In [ ]:
# Build a DATE-indexed equity curve for the LLM strategy so it can be plotted and
# scored on the same footing as the baselines. This needs a date on each closed
# trade; if none of these keys exist the code falls back to a per-trade curve.
DATE_KEYS = ("exit_date", "close_date", "closed_at", "exit_time", "date")

def _to_day(value):
    ts = pd.Timestamp(value)
    if ts.tzinfo is not None:
        ts = ts.tz_convert("UTC").tz_localize(None)
    return ts.normalize()

def llm_daily_equity(trades, index, capital):
    by_day = {}
    for tr in trades:
        raw = next((tr[k] for k in DATE_KEYS if tr.get(k) is not None), None)
        if raw is None:
            return None
        day = _to_day(raw)
        by_day[day] = by_day.get(day, 0.0) + float(tr.get("pnl", 0.0))
    idx = pd.DatetimeIndex(index)
    if idx.tz is not None:
        idx = idx.tz_localize(None)
    idx = idx.normalize()
    if not by_day:
        return pd.Series(capital, index=idx)
    cum = pd.Series(by_day).sort_index().cumsum()
    full = cum.reindex(cum.index.union(idx)).ffill().reindex(idx).fillna(0.0)
    return capital + full

# per ticker: equity/returns/pnls for each of the three strategies
curves = {}
for t, base in baselines.items():
    trades = llm_sims.get(t, {}).get("closed_trades", [])
    llm_pnls = [tr.get("pnl", 0.0) for tr in trades]
    bh_equity = base["Buy-and-hold"].portfolio_value
    llm_equity = llm_daily_equity(trades, bh_equity.index, starting_capital)
    dated = llm_equity is not None
    if not dated:
        curve = [starting_capital]
        for p in llm_pnls:
            curve.append(curve[-1] + p)
        llm_equity = pd.Series(curve)
        print(f"{t}: no date on closed trades -> LLM Sharpe is per-trade, NOT comparable.")
    assert llm_equity is not None
    curves[t] = {
        "Buy-and-hold": (base["Buy-and-hold"].portfolio_value, base["Buy-and-hold"].returns, trade_pnls(base["Buy-and-hold"])),
        "SMA 50/200":   (base["SMA 50/200"].portfolio_value,   base["SMA 50/200"].returns,   trade_pnls(base["SMA 50/200"])),
        "LLM-guided":   (llm_equity, llm_equity.pct_change().fillna(0.0), llm_pnls),
        "_llm_dated": dated,
    }

In [ ]:
def summarise(equity, returns, pnls):
    equity = pd.Series(np.asarray(equity, dtype=float))
    return {
        "Sharpe":       compute_sharpe(returns) if len(returns) else np.nan,
        "Max Drawdown": compute_max_drawdown(equity) if len(equity) > 1 else np.nan,
        "Win Rate":     compute_win_rate(pnls) if pnls else np.nan,
        "Trades":       len(pnls),
        "Total Return": equity.iloc[-1] / starting_capital - 1,
        "Final Equity": equity.iloc[-1],
    }

records = {}
for t, c in curves.items():
    for strat in ("Buy-and-hold", "SMA 50/200", "LLM-guided"):
        records[(t, strat)] = summarise(*c[strat])
metrics = pd.DataFrame(records).T.astype(float)
metrics.index.names = ["Ticker", "Strategy"]

def _fmt(df):
    out = df.copy()
    out["Sharpe"]       = df["Sharpe"].map(lambda v: "N/A" if pd.isna(v) else f"{v:.3f}")
    out["Max Drawdown"] = df["Max Drawdown"].map(lambda v: "N/A" if pd.isna(v) else f"{v:.2%}")
    out["Win Rate"]     = df["Win Rate"].map(lambda v: "N/A" if pd.isna(v) else f"{v:.2%}")
    out["Trades"]       = df["Trades"].astype(int)
    out["Total Return"] = df["Total Return"].map(lambda v: f"{v:.2%}")
    out["Final Equity"] = df["Final Equity"].map(lambda v: f"${v:,.2f}")
    return out

print("=== Strategy Comparison ===")
display(_fmt(metrics))

In [ ]:
n = max(len(curves), 1)
fig, axes = plt.subplots(n, 1, figsize=(12, 4.5 * n), squeeze=False)
for ax, (t, c) in zip(axes[:, 0], curves.items()):
    bh = c["Buy-and-hold"][0]
    ax.plot(bh.index, bh.to_numpy(dtype=float), label="Buy-and-hold", linewidth=2)
    sma = c["SMA 50/200"][0]
    ax.plot(sma.index, sma.to_numpy(dtype=float), label="SMA 50/200", linewidth=2)
    if c["_llm_dated"]:
        ax.plot(bh.index, c["LLM-guided"][0].to_numpy(dtype=float),
                label="LLM-guided (realised)", linewidth=2)
    else:
        ax.set_title(f"{t} (LLM curve omitted: trades have no dates)")
    if c["_llm_dated"]:
        ax.set_title(f"Equity Curves - {t}")
    ax.set_ylabel("Portfolio Value ($)")
    ax.legend(); ax.grid(True)
axes[-1, 0].set_xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
report_dir = ROOT / "reports"
report_dir.mkdir(exist_ok=True)
report_path = report_dir / "evaluation_report.md"

table = _fmt(metrics)
lines = [
    "# Evaluation Report", "",
    f"**Period:** {start_date} to {end_date}  ",
    f"**Tickers:** {', '.join(curves)}  ",
    f"**Starting Capital (per ticker/strategy):** ${starting_capital:,.2f}  ", "",
    "## Performance Metrics", "",
    "| Ticker | Strategy | Sharpe | Max Drawdown | Win Rate | Trades | Total Return | Final Equity |",
    "|---|---|---|---|---|---|---|---|",
]
for t, strat, sharpe, max_drawdown, win_rate, trades, total_return, final_equity in (
    table.reset_index().itertuples(index=False, name=None)
):
    lines.append(
        f"| {t} | {strat} | {sharpe} | {max_drawdown} | {win_rate} "
        f"| {trades} | {total_return} | {final_equity} |"
    )
lines += [
    "", "## Observations", "",
    "1. **Buy-and-hold** is the passive benchmark.",
    "2. **SMA 50/200** is a simple trend-following rule.",
    "3. **LLM-guided** uses the agent's daily signals (news sentiment + LLM analysis). "
    "It is scored on realised P&L of closed trades only.",
    "", "## Conclusion", "",
]
for t in curves:
    sub = metrics.loc[t]
    best = sub["Total Return"].idxmax()
    lines.append(
        f"- **{t}:** highest total return was **{best}** "
        f"({sub.loc[best, 'Total Return']:.2%}); LLM-guided returned "
        f"{sub.loc['LLM-guided', 'Total Return']:.2%} vs "
        f"{sub.loc['Buy-and-hold', 'Total Return']:.2%} for buy-and-hold."
    )
lines += ["", "One window and a few tickers is not evidence of a durable edge; "
          "repeat across more windows and tickers."]
report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Report written to {report_path}")